Kilter Board: Data Overview and Climbing Statistics

## Purpose

This notebook establishes the basic statistical landscape of the dataset before we move into hold-level analysis and predictive modelling. The main goals are:

1. to understand the size and scope of the data,
2. to compare layouts, boards, and angles at a high level,
3. to identify broad trends in grade, popularity, and quality,
4. to create a clean descriptive baseline for the later modelling notebooks.

Throughout, I treat each climb-angle entry as a separate observation unless explicitly noted otherwise. That matters because some climbs appear at multiple angles, so a unique climb count and a climb-angle count are not always the same thing.

## Outputs

This notebook produces summary tables and exploratory plots that motivate the later notebooks on:
- hold usage,
- hold difficulty,
- feature engineering,
- predictive modelling,
- and deep learning.

## Notebook Structure
1. [Setup and Imports](#setup-and-imports)
2. [Popularity and Temporal Trends](#popularity-and-temporal-trends)
3. [Climbing Statistics](#climbing-statistics-grades-angles-quality-and-matching)
4. [Prolific Statistics](#prolific-statistics)
5. [Conclusion](#conclusion)

## Setup and Imports

In [ ]:
"""
==================================
Setup and imports
==================================
"""
# Imports
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import matplotlib.patches as mpatches
import sqlite3


# Set some display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set style
palette=['steelblue', 'coral', 'seagreen']  #(for multi-bar graphs)

# Connect to the database
DB_PATH="../data/kilter.db"
conn = sqlite3.connect(DB_PATH)


In [ ]:
"""
==================================
### Query our data from the DB
==================================

We restrict ourselves to the Kilter Original, i.e., layoud_id=1.
For some reason there is also a first ascent logged in 2006 ("'Preliminary first draft of Val David Sans Nom Practice. Not yet tested").
Since the first logged ascents start in 2018, we'll just insist that the date is larger than January 1, 2016.
"""

# Query climb data
climbs_query = """
SELECT
    c.uuid,
    c.name AS climb_name,
    c.setter_username,
    c.layout_id AS layout_id,
    c.description,
    c.is_nomatch,
    c.is_listed,
    l.name AS layout_name,
    p.name AS board_name,
    c.frames,
    cs.angle,
    cs.display_difficulty,
    dg.boulder_name AS boulder_grade,
    cs.ascensionist_count,
    cs.quality_average,
    cs.fa_at
FROM climbs c
JOIN layouts l ON c.layout_id = l.id
JOIN products p ON l.product_id = p.id
JOIN climb_stats cs ON c.uuid = cs.climb_uuid
JOIN difficulty_grades dg ON ROUND(cs.display_difficulty) = dg.difficulty
WHERE cs.display_difficulty IS NOT NULL AND c.layout_id=1 AND cs.fa_at > '2016-01-01';
"""

# Load it into a DataFrame
df = pd.read_sql_query(climbs_query, conn)

The above query will allow us to gather basically anything we need to in order to analyze climbing statistics. We leave out information about climging holds and things like this, because they will be analyzed in a different notebook. Let's see what our DataFrame looks like.

In [ ]:
df

---

# Popularity and Temporal Trends

## Popularity of Tension Board

Since we do not have access to user data, we will examine the popular of the Tension Boards by counting first ascents and unique setters by year. Often it's the case that the first ascensionist is the also the setter of the climb, but not always. None the less, we group up first ascensionists by year, with an extra tidbit about how many unique setters there were. 

In [ ]:
"""
==================================
Popularity of Kilter board by year. 
First ascents by year + unique setters by year
==================================
"""

# Convert df['fa_at'] to datetime format. (For some reason, it does not register as such)
df['fa_at'] = pd.to_datetime(df['fa_at'])

# Add a new column for the year
df['fa_year'] = df['fa_at'].dt.year

# Make a new DataFrame with year, first_ascents, and unique_setters
df_growth = df.groupby('fa_year').agg(
    first_ascents=('uuid', 'count'),
    unique_setters=('setter_username', 'nunique')
).reset_index()

# Disregard the year 2026 since the data only goes one month in. 
df_growth = df_growth[df_growth['fa_year'] < 2026]

# Convert year to string so that matplotlib doesn't think the year is continuous
df_growth['fa_year'] = df_growth['fa_year'].astype(str)

## Plot
# Dual index plotting

fig, ax1 = plt.subplots(figsize=(12,6))

# Bar chart for first ascents
ax1.bar(df_growth['fa_year'], df_growth['first_ascents'], label='First Ascents', color='coral')
ax1.set_xlabel('Year')
ax1.set_ylabel('First Ascents')
ax1.set_title('TB First Ascents & Unique Setters over Time')
#ax1.tick_params(axis='y')

# Line chart for unique setters (secondary axis)
ax2 = ax1.twinx()
ax2.plot(df_growth['fa_year'], df_growth['unique_setters'], color='steelblue', marker='o', label='Unique Setters')
ax2.set_ylabel('Unique Setters', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')

# Other stuff
fig.legend(loc='upper left', bbox_to_anchor=(0.15,0.85))

plt.xticks()
plt.savefig('../images/01_climb_stats/first_ascents_by_year.png')
plt.show()

## Seasonal analysis

Next, we examine when the Tension board is most popular. Again, we will work with what we have and use first ascent data. We will plot first ascents by month, combing all years. We exclude the year 2026 because this can skew the analysis as some of the month of January has data (and clearly, 2026 is when the TB2 is the most popular, so this can actually add quite a bit bias). 

In [ ]:
"""
==================================
Season analysis: first ascents by month
==================================
"""

# First let us add a column for the month to our data
df['fa_month'] = df['fa_at'].dt.month

# Filter to years < 2026 since the data only goes one month in
df_filter = df[df['fa_year'] < 2026]

# Make a new DataFrame with month and first ascents
df_season = df_filter.groupby('fa_month').agg(
    first_ascents=('uuid', 'count'),
).reset_index()

# We also add a column for the month name. 
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df_season['fa_month_name'] = df_season['fa_month'].apply(lambda x: month_names[x-1])

# Plot the data
fig,ax = plt.subplots(figsize=(12,6))
ax.bar(df_season['fa_month_name'], df_season['first_ascents'], color='coral')
ax.set_title('First Ascents by Month (All Years Combined)')
ax.set_xlabel('Month')
ax.set_ylabel('Total First Ascents')

# Save the file
plt.savefig('../images/01_climb_stats/first_ascents_by_month.png')
plt.show()

This should be what we expect: that the winter months (Dec) see the most traffic. This is probably when the outdoor climbers are hitting the boards because they're stuck inside. The warmer months see the least number of first ascents since the strong climbers are probably outdoors.

## Day of Week Analysis

We can plot the number of first ascents by day of week. Removing the 2026 data shouldn't make a difference here, so we opt to keep it.

In [ ]:
"""
==================================
Day of Week analysis
==================================
"""

# Let us add a column in our DataFrame for the day of the week.
# Note that df.dt.day_of_week will have Monday be 0 and Sunday be 6. 

df['fa_day_of_week'] = df['fa_at'].dt.day_of_week


# Make a new DataFrame with month and first ascents
df_days = df.groupby('fa_day_of_week').agg(
    first_ascents=('uuid', 'count'),
).reset_index()

# We also add a column for the month name. 
day_names = ['Mon', 'Tues', 'Wed', 'Thurs', 'Fri', 'Sat', 'Sun']
df_days['fa_day_name'] = df_days['fa_day_of_week'].apply(lambda x: day_names[x])

# Plot the data
fig,ax = plt.subplots(figsize=(12,6))
ax.bar(df_days['fa_day_name'], df_days['first_ascents'], color='coral')
ax.set_title('First Ascents by Day of Week (All Years Combined)')
ax.set_xlabel('Day')
ax.set_ylabel('Total First Ascents')

# Save the file
plt.savefig('../images/01_climb_stats/first_ascents_by_day_of_week.png')
plt.show()



Interesting, Tuesday and Wednesday have the most traffic, while Monday is the least popular.

## Time of Day Analysis

We can even do a time of day analysis. Again, we will keep the 2026 data since it shouldn't affect much. It is not entirely clear that makes sense to look at this, as we don't know if the time of first ascent is recorded in local time of the climber or local time of the server. These boards are all over the world, so this may add quite a bit of variance.

In [ ]:
"""
==================================
Time of Day analysis
==================================
"""

# Let us add a column in our DataFrame for the day of the week.
# Note that df.dt.day_of_week will have Monday be 0 and Sunday be 6. 

df['fa_hour'] = df['fa_at'].dt.hour


# Make a new DataFrame with month and first ascents
df_hour = df.groupby('fa_hour').agg(
    first_ascents=('uuid', 'count'),
).reset_index()


# Plot the data
fig,ax = plt.subplots(figsize=(12,6))
ax.bar(df_hour['fa_hour'], df_hour['first_ascents'], color='coral')
ax.set_title('First Ascents by Hour (All Years Combined)')
ax.set_xlabel('Hour')
ax.set_ylabel('Total First Ascents')

# Save the file
plt.savefig('../images/01_climb_stats/first_ascents_by_hour.png')
plt.show()



---

# Climbing Statistics: Grades, Angles, Quality, and Matching

We will visualize the climbing grade distribution. Recall that we have the following table of grades (with some other unlisted grades).

|difficulty|boulder_name|route_name|
|----------|------------|----------|
|        10|4a/V0       |5b/5.9    |
|        11|4b/V0       |5c/5.10a  |
|        12|4c/V0       |6a/5.10b  |
|        13|5a/V1       |6a+/5.10c |
|        14|5b/V1       |6b/5.10d  |
|        15|5c/V2       |6b+/5.11a |
|        16|6a/V3       |6c/5.11b  |
|        17|6a+/V3      |6c+/5.11c |
|        18|6b/V4       |7a/5.11d  |
|        19|6b+/V4      |7a+/5.12a |
|        20|6c/V5       |7b/5.12b  |
|        21|6c+/V5      |7b+/5.12c |
|        22|7a/V6       |7c/5.12d  |
|        23|7a+/V7      |7c+/5.13a |
|        24|7b/V8       |8a/5.13b  |
|        25|7b+/V8      |8a+/5.13c |
|        26|7c/V9       |8b/5.13d  |
|        27|7c+/V10     |8b+/5.14a |
|        28|8a/V11      |8c/5.14b  |
|        29|8a+/V12     |8c+/5.14c |
|        30|8b/V13      |9a/5.14d  |
|        31|8b+/V14     |9a+/5.15a |
|        32|8c/V15      |9b/5.15b  |
|        33|8c+/V16     |9b+/5.15c |

We will use the actual difficulty in our work, and then unpack translations into boulder_name as we see fit.

## Grade distribution

In [ ]:
"""
==================================
Difficulty distribution
==================================
"""

grade_counts = df['boulder_grade'].value_counts()
grade_order = df.groupby('boulder_grade')['display_difficulty'].mean().sort_values().index.tolist()
grade_counts = grade_counts.reindex(grade_order)


df_grades = df.groupby('boulder_grade').size().reset_index(name='count')


# Plot
fig, ax = plt.subplots(figsize=(16, 8))

sns.barplot(
    data=df_grades,
    x='boulder_grade',
    y='count',
    color='steelblue',
    ax=ax,
    width=0.6,
    order=grade_order
)



ax.set_xlabel('Grade', fontsize=11)
ax.set_ylabel('Number of Climbs', fontsize=11)
ax.set_title('Difficulty Distribution by Board Layout', fontsize=14)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/01_climb_stats/difficulty_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

As a climber in North America, I tend to just use the V-grade and not look at the French grade. So let us group the V-grades together and show the distribution like that. We'll usually just stick the boulder_grade (e.g., 5c/V2) instead of grouping the V-grades though. 

In [ ]:
"""
==================================
V-Grade distribution
==================================
"""

grade_to_v = {
10: 0, 11: 0, 12: 0,
13: 1, 14: 1,
15: 2,
16: 3, 17: 3,
18: 4, 19: 4,
20: 5, 21: 5,
22: 6,
23: 7,
24: 8, 25: 8,
26: 9,
27: 10,
28: 11,
29: 12,
30: 13,
31: 14,
32: 15,
33: 16,
}

# Let's add a v_grade column and v_grade_counts
df['v_grade'] = df['display_difficulty'].round().map(grade_to_v)
df_v_grades = df.groupby('v_grade').size().reset_index(name='count')
df_v_grades['v_label'] = 'V' + df_v_grades['v_grade'].astype(str)


# Plot
fig, ax = plt.subplots(figsize=(16, 8))

sns.barplot(
    data=df_v_grades,
    x='v_label',
    y='count',
    color='steelblue',
    ax=ax,
    width=0.6,
)


ax.set_xlabel('V-Grade', fontsize=11)
ax.set_ylabel('Number of Climbs', fontsize=11)
ax.set_title('V-Grade Distribution by Board Layout', fontsize=14)
ax.tick_params(axis='x')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/01_climb_stats/v_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Some key differences in grades are the angle at which the climb is. Note that climbs can be done at different angles.

## Angle Distribution

What about the angle distribution? Since the TB1 goes from 0 to 50 and the TB2 goes from 0 to 65 (although my local board only goes to 60?), let's do an analysis on each.

In [ ]:
"""
==================================
Angle distribution
==================================

The Kilter Board original goes to 70 degrees.
"""


df_angle = df.groupby('angle').size().reset_index(name='count')

# Reindex to correct order
angle_order = sorted(df['angle'].unique())

# Plot
fix, ax = plt.subplots(figsize=(16,8))

# Plot All Layouts
sns.barplot(
    data=df_angle,
    x='angle',
    y='count',
    color='seagreen',
    ax=ax,
    width=0.6,
    order=angle_order
)


ax.set_xlabel('Angle')
ax.set_ylabel('Number of Climbs')
ax.set_title('Angle Distribution by Board Layout')
ax.grid(axis='y', alpha=0.3)


plt.suptitle('Angle Distribution by Board Layout')
plt.savefig('../images/01_climb_stats/angle_distribution.png')
plt.show()

Just like with Tension Boards, 40 is the most common angle. 

## Angle vs grade

How is the distribution between angles and grades? Let's do this with a heatmap.

In [ ]:
"""
==================================
Angle vs grade
==================================
"""

fig, ax = plt.subplots(figsize=(16, 8))

# Create mapping from difficulty to boulder_grade
grade_mapping = df.groupby('display_difficulty')['boulder_grade'].first().to_dict()

# Plot "All Layouts" as faint background boxes
sns.boxplot(
    data=df,
    x='angle',
    y='display_difficulty',
    color='seagreen',
    order=angle_order,
    showfliers=False,
    width=0.6,
    ax=ax,
)

# Relabel y-axis with boulder_grades
yticks_rounded = sorted(set(int(round(t)) for t in df['display_difficulty'].unique() if not pd.isna(t)))
ylabels = [grade_mapping.get(t, '') for t in yticks_rounded]
ax.set_yticks(yticks_rounded)
ax.set_yticklabels(ylabels)


ax.set_xlabel('Angle (degrees)', fontsize=11)
ax.set_ylabel('Boulder Grade', fontsize=11)
ax.set_title('Difficulty Distribution by Angle', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/01_climb_stats/difficulty_by_angle_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

We see that angle is directly correlated with how difficult climbs are on average, right up untill 55 degrees. Then it tapers off.

## The Quality of a climb

Next we examine the quality of a climb. First we look at how quality relates to the number of ascents.

In [ ]:
"""
==================================
Climb quality vs popularity
==================================
"""

# Filter to climbs with quality ratings
df_quality = df[(df['quality_average'].notna()) & (df['quality_average'] > 0)]

# Sample for performance
#df_sample = df_quality.sample(min(2000, len(df_quality)))

g = sns.jointplot(
    data=df_quality,
    x='quality_average',
    y='ascensionist_count',
    kind='scatter',
    color='teal',
    height=5
)

g.ax_joint.set_xlabel('Quality Rating')
g.ax_joint.set_ylabel('Ascensionist Count')
g.fig.suptitle('Quality vs Popularity')

plt.savefig('../images/01_climb_stats/quality_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

Next we visualize the average quality vs the angle and grade, by means of a heatmap. Keep in mind that the harder the climb and steeper the angle, the less people will be doing it. So harder climbs are skewed towards people who can actually do it. The point is that, on boards, the climb quality isn't always the best metric. As such, we won't spend too much time on the quality and will only do a heatmap which takes into account all layouts.

In [ ]:
### Average quality by angle and grade

# Filter to climbs with quality ratings
df_quality = df[(df['quality_average'].notna()) & (df['quality_average'] > 0)]


# Create pivot table
quality_pivot = df_quality.pivot_table(
    index='boulder_grade',
    columns='angle',
    values='quality_average',
    aggfunc='mean'
)
quality_pivot = quality_pivot.reindex(grade_order)
quality_pivot = quality_pivot.reindex(columns=[a for a in angle_order if a in quality_pivot.columns])

# Plot
fig, ax = plt.subplots(figsize=(16, 8))

sns.heatmap(
    quality_pivot,
    cmap='RdYlGn',
    cbar_kws={'label': 'Avg Quality Rating'},
    ax=ax
)

ax.set_xlabel('Angle (°)')
ax.set_ylabel('Grade')
ax.invert_yaxis()
ax.set_title('Average Quality Rating by Grade and Angle (All Layouts)')

plt.tight_layout()
plt.savefig('../images/01_climb_stats/quality_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## "Match" vs. "No Match"

Some setters opt to put the "no match" tag onto their climbs. This means that the climber is not allowed to match their hands on any hold. Let's do an analysis of the differences with regular climbs.

In [ ]:
"""
==================================
Match vs No Match analysis
==================================
"""

# Create status column
df['status'] = df.apply(
    lambda x: 'No Match' if (
        pd.notna(x['description']) and 'No matching' in str(x['description'])
    ) or x.get('is_nomatch', 0) == 1 else 'Matched',
    axis=1
)

# Aggregate by status only
df_agg = df.groupby('status').agg(
    count=('uuid', 'count'),
    avg_ascensionists=('ascensionist_count', 'mean'),
    avg_difficulty=('display_difficulty', 'mean')
).reset_index()

status_order = ['Matched', 'No Match']
status_colors = {'Matched': 'teal', 'No Match': 'coral'}

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(axes, ['count', 'avg_difficulty', 'avg_ascensionists'], 
                               ['Total Climbs', 'Average Difficulty', 'Avg Ascensionists']):
    
    sns.barplot(
        data=df_agg,
        x='status',
        y=metric,
        hue='status',
        legend=False,
        order=status_order,
        palette=status_colors,
        ax=ax,
        width=0.5
    )
    
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

# Y-axis labels for difficulty plot
yticks = [11, 13, 15, 17, 19, 21, 23]
ylabels = [grade_mapping.get(t, f"V{t-10}") for t in yticks]
axes[1].set_yticks(yticks)
axes[1].set_yticklabels(ylabels)
axes[1].set_ylim(bottom=10)

# Add value labels on bars
for ax in axes:
    for p in ax.patches:
        if ax == axes[1]:  # Difficulty plot - show boulder_grade
            height = p.get_height()
            rounded_diff = round(height)
            boulder_grade = grade_mapping.get(rounded_diff, f"V{rounded_diff - 10}")
            ax.annotate(
                boulder_grade,
                (p.get_x() + p.get_width() / 2, height),
                ha='center',
                va='bottom',
                fontsize=10,
                fontweight='bold'
            )
        else:  # Other plots - show numeric values
            fmt = f'{p.get_height():,.0f}' if ax == axes[0] else f'{p.get_height():.1f}'
            ax.annotate(
                fmt,
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center',
                va='bottom',
                fontsize=10
            )

plt.suptitle('Match vs No Match Climbs (All Layouts)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../images/01_climb_stats/match_vs_nomatch.png', dpi=150, bbox_inches='tight')
plt.show()

So we gather the following about "no match" climbs:

- they are far fewer than "match" climbs,
- they are on average harder than "match" climbs,
- and that they have quite a bit less ascensionists on average.

In [ ]:
"""
==================================
Match vs No Match Summary
==================================
"""


summary = df_agg.pivot_table(
    columns='status',
    values=['count', 'avg_difficulty', 'avg_ascensionists']
).round(2)

summary

---

# Prolific statistics

Here we will take note of some prolific statistics: what are the most popular climbs and who are the most popular setters?

## Most popular climbs

In [ ]:
"""
==================================
Most popular climbs
==================================
"""

# The ascensionist_count column will allow us to easily deduce the top 15 climbs. 

# Create a DataFrame with the top 15 climbs
df_popular_climbs = df.sort_values(by='ascensionist_count', ascending=False).head(15).reset_index(drop=True)

# Appropriate index
df_popular_climbs.index = df_popular_climbs.index + 1


display(df_popular_climbs[['climb_name', 'setter_username', 'angle', 'boulder_grade', 'ascensionist_count']])


It's unsuprising that every one of these climbs is at 40° given that 40° is the most popular angle, by a long shot.

What about an angle-agnostic analysis? What are the top climbs amonst all angles?

In [ ]:
"""
==================================
Top 15 most popular climbs (angle agnostic)
==================================
"""

# Aggregate by climb_name (sum counts across all angles)
df_agg = df.groupby(['climb_name']).agg(
    total_ascensionists=('ascensionist_count', 'sum'),
    avg_difficulty=('display_difficulty', 'mean')
).reset_index()


df_agg['avg_boulder_grade'] = df_agg['avg_difficulty'].round().astype(int).map(grade_mapping)

# Sort and select top 15
df_popular_climbs_aa = df_agg.sort_values(by='total_ascensionists', ascending=False).head(15).reset_index(drop=True)

df_popular_climbs_aa.index = df_popular_climbs_aa.index + 1

display(df_popular_climbs_aa)



## Prolific setters

Next, we will make a simple table of the most prolific setters by board.

In [ ]:
"""
==================================
Top 10 setters
==================================
"""

# Make a DataFrame for the setters
df_agg = df.groupby(['setter_username']).agg(
    climb_count=('uuid', 'nunique')
).reset_index()

df_setters = df_agg.sort_values(by='climb_count', ascending=False).head(10).reset_index(drop=True)

df_setters.index = df_setters.index + 1

display(df_setters)


---

# Conclusion

At this point we have a board-level and climb-level picture of the dataset. In particular, we now know:

- how large the dataset is,
- how the grade and angle distributions vary across layouts,
- which climbs and setters appear most often,
- and where simple descriptive trends begin to show up.

That gives us enough context to move from *global statistics* to *hold-level structure*. The next notebook focuses on hold usage patterns and board heatmaps, where we stop asking only **how many climbs there are** and start asking **which physical parts of the board are driving those climbs**.